# INFIDRUG MOLEQULAR FUSION — Gradient Boosting Regression (GB)
#
# Team TESSARACT. Target: measured log-solubility (mol/L) on delaney.csv.
# Matched benchmark: 256 SMILES features -> 4-component TruncatedSVD ->
# training-fitted scaling. A separate 256-feature benchmark is included.

## Notebook 1 — Gradient Boosting Regression (GB)
#
# Classical pipeline: Delaney loading, group-aware splitting, SVD reduction
# (256 -> 4 features), hyperparameter search, full 256-feature benchmark,
# inference on new molecules and ROC analysis.

### Cell 1 — Imports & library setup

In [ ]:
code
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

### Cell 2 — Dataset loading and group-aware split (train / val / test)

In [ ]:
code
# Load data
df = pd.read_csv("delaney.csv")

# Automatically find the required columns
SMILES_COL = next(c for c in df.columns if "smiles" in c.lower())
TARGET_COL = next(c for c in df.columns
                  if "measured" in c.lower() and "solubility" in c.lower())

df = df.dropna(subset=[SMILES_COL, TARGET_COL]).copy()
smiles = df[SMILES_COL].astype(str)
y = df[TARGET_COL].astype(float).to_numpy()

print("Dataset shape:", df.shape)
print("SMILES column:", SMILES_COL)
print("Target column:", TARGET_COL)
display(df[[SMILES_COL, TARGET_COL]].head())

# OUTER split -> final test set
groups = smiles.to_numpy()
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
dev_idx, test_idx = next(outer.split(smiles, y, groups))

# INNER split -> train and validation
inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43)
tr_rel, val_rel = next(
    inner.split(smiles.iloc[dev_idx], y[dev_idx], groups[dev_idx])
)

train_idx = dev_idx[tr_rel]
val_idx = dev_idx[val_rel]

y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

### Cell 3 — Exploratory data analysis

In [ ]:
code
# Basic EDA
print(df[[TARGET_COL]].describe())
print("\nMissing values:", df[[SMILES_COL, TARGET_COL]].isna().sum().to_dict())
print("Duplicate SMILES:", df[SMILES_COL].duplicated().sum())

plt.figure(figsize=(7, 4))
plt.hist(y, bins=30, edgecolor="black")
plt.xlabel("Measured logS")
plt.ylabel("Count")
plt.title("Target Distribution")
plt.show()

### Cell 4 — Feature preparation (CountVectorizer + TruncatedSVD + scaling)

In [ ]:
code
# SMILES -> 256 character-count features
vectorizer = CountVectorizer(analyzer="char", max_features=256)
X256_train = vectorizer.fit_transform(smiles.iloc[train_idx])
X256_val = vectorizer.transform(smiles.iloc[val_idx])
X256_test = vectorizer.transform(smiles.iloc[test_idx])
print("256-feature shapes:",
      X256_train.shape, X256_val.shape, X256_test.shape)

# 256 -> 4 components for matched benchmark
svd = TruncatedSVD(n_components=4, random_state=42)
X4_train = svd.fit_transform(X256_train)
X4_val = svd.transform(X256_val)
X4_test = svd.transform(X256_test)

# Scale using TRAINING data only
scaler = StandardScaler()
X4_train = scaler.fit_transform(X4_train)
X4_val = scaler.transform(X4_val)
X4_test = scaler.transform(X4_test)

print("4-feature shapes:", X4_train.shape, X4_val.shape, X4_test.shape)
print("Variance retained by 4 components:",
      round(svd.explained_variance_ratio_.sum() * 100, 2), "%")

### Cell 5 — Regression metrics & plotting helper

In [ ]:
code
def regression_metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred),
    }


def show_evaluation(name, y_true, pred):
    m = regression_metrics(y_true, pred)
    display(pd.DataFrame([{"Model": name, **m}]))

    # Actual vs Predicted
    plt.figure(figsize=(6, 5))
    plt.scatter(y_true, pred, alpha=0.65)
    lo = min(y_true.min(), pred.min())
    hi = max(y_true.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], "r--")
    plt.xlabel("Actual logS")
    plt.ylabel("Predicted logS")
    plt.title(name + " - Actual vs Predicted")
    plt.show()

    # Residual plot
    residuals = y_true - pred
    plt.figure(figsize=(6, 4))
    plt.scatter(pred, residuals, alpha=0.65)
    plt.axhline(0, color="red", linestyle="--")
    plt.xlabel("Predicted logS")
    plt.ylabel("Residual")
    plt.title(name + " - Residual Plot")
    plt.show()
    return m

### Cell 6 — Validation-based model selection (tuning GB candidates)

In [ ]:
code
candidates = {
    "GB-1": GradientBoostingRegressor(n_estimators=100, learning_rate=0.05,
                                      max_depth=2, random_state=42),
    "GB-2": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                      max_depth=2, random_state=42),
    "GB-3": GradientBoostingRegressor(n_estimators=200, learning_rate=0.03,
                                      max_depth=3, random_state=42),
}

rows = []
trained = {}
for name, model in candidates.items():
    model.fit(X4_train, y_train)
    pred_val = model.predict(X4_val)
    m = regression_metrics(y_val, pred_val)
    rows.append({"Candidate": name, **m})
    trained[name] = model

validation_results = pd.DataFrame(rows).sort_values("RMSE")
display(validation_results)

best_name = validation_results.iloc[0]["Candidate"]
best_model = trained[best_name]
print("Selected using lowest validation RMSE:", best_name)

### Cell 7 — Final matched 4-feature test evaluation

In [ ]:
code
test_pred = best_model.predict(X4_test)
matched_metrics = show_evaluation(
    "Gradient Boosting Regression - 4 features", y_test, test_pred
)

### Cell 8 — Separate full 256-feature classical benchmark

In [ ]:
code
full_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                       max_depth=2, random_state=42)
full_model.fit(X256_train, y_train)

full_val_pred = full_model.predict(X256_val)
print("256-feature validation metrics:")
display(pd.DataFrame([regression_metrics(y_val, full_val_pred)]))

full_test_pred = full_model.predict(X256_test)
full_metrics = show_evaluation(
    "Gradient Boosting Regression - 256 features", y_test, full_test_pred
)

### Cell 9 — 4-feature vs 256-feature comparison table

In [ ]:
code
comparison = pd.DataFrame([
    {"Experiment": "Matched 4 features", **matched_metrics},
    {"Experiment": "Full 256 features", **full_metrics},
])
display(comparison)

### Cell 10 — Predict new SMILES molecules

In [ ]:
code
new_smiles = ["CCO", "c1ccccc1"]

new_256 = vectorizer.transform(new_smiles)
new_4 = scaler.transform(svd.transform(new_256))
new_pred = best_model.predict(new_4)

pd.DataFrame({
    "SMILES": new_smiles,
    "Predicted logS": new_pred,
})

### Cell 11 — ROC analysis & prediction export

In [ ]:
code
# Extra ROC analysis (main task is still regression)
# Positive class: logS >= -2.0
from sklearn.metrics import roc_curve, roc_auc_score

ROC_THRESHOLD = -2.0
y_binary = (np.asarray(y_test) >= ROC_THRESHOLD).astype(int)

# Find the test prediction made above
roc_pred = None
for n in ["test_pred", "y_pred", "pred_test", "test_prediction", "pred", "y_test_pred"]:
    if n in globals():
        a = np.asarray(globals()[n]).reshape(-1)
        if len(a) == len(y_test):
            roc_pred = a
            break

if roc_pred is None:
    raise NameError("Run the model cells above first; test predictions were not found.")

fpr, tpr, _ = roc_curve(y_binary, roc_pred)
auc_value = roc_auc_score(y_binary, roc_pred)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, lw=2, label=f"Model (AUC = {auc_value:.4f})")
plt.plot([0, 1], [0, 1], "--", label="Random Guess (AUC = 0.5000)")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title(f"ROC Curve (Threshold: logS >= {ROC_THRESHOLD})")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.show()

print("ROC-AUC:", round(auc_value, 4))
np.save("gb_pred.npy", test_pred)